# ROQ Parameter Estimation for GW170817 — Optimized

Parameter estimation using the **mlgw\_bns\_jax** surrogate waveform model
with **Reduced Order Quadrature (ROQ)** likelihood acceleration and
the **SHARPy** SMC sampler.

**Optimized version** — uses `jax_predict_roq.py` and `roq_likelihood_mlgw_bns_jax.py`
so that PN terms are evaluated **only at the N\_ROQ ≈ 224 nodes** instead of on
the model's internal ~6 200-pt grid. This yields the true ROQ speedup:

| Step | Original | Optimized |
|------|----------|-----------|
| MLP + PCA forward | O(fixed) | O(fixed) — unchanged |
| PN amplitude/phase | O(6 200) per call | **O(N\_ROQ) ≈ 224** per call |
| Spline to query freqs | O(253 k) | **O(N\_ROQ)** |

**Prerequisites:** Run `build_roq_basis_numpy.ipynb` (or `build_roq_basis_numpy.py`)
first to generate the ROQ interpolants in `roq_basis_mlgw_bns_jax/ROQ_data/`.

**New supporting files** (do not edit the originals):
- `jax_predict_roq.py` — `load_predict_nodes()` function
- `roq_likelihood_mlgw_bns_jax.py` — `build_roq_template()` and `build_roq_likelihood()`

In [ ]:
import os, subprocess, sys

COLAB = "google.colab" in sys.modules
IGWN  = os.path.exists("/cvmfs/oasis.opensciencegrid.org")
REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

REPO_GIT    = "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git"
REPO_BRANCH = "copilot/add-reduced-order-quadrature-model"
SHARPY_GIT  = "https://github.com/gabrieledemasi/sharpy.git"

if COLAB:
    print("=== Google Colab detected ===")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "corner", "gwpy", "h5py", "ripplegw", "lalsuite", "jaxopt", "netket",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps",
        "blackjax @ git+https://github.com/gabrieledemasi/blackjax@main",
    ])

    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
            REPO_GIT, REPO_DIR,
        ])

    sharpy_repo = os.path.join(REPO_DIR, "_sharpy_repo")
    sharpy_pkg  = os.path.join(sharpy_repo, "sharpy")
    sharpy_link = os.path.join(REPO_DIR, "sharpy")
    if not os.path.isdir(sharpy_repo):
        subprocess.check_call(["git", "clone", "--depth", "1", SHARPY_GIT, sharpy_repo])
    if not os.path.exists(sharpy_link):
        os.symlink(sharpy_pkg, sharpy_link)

    import site
    for _sp in site.getsitepackages():
        _chees = os.path.join(_sp, "blackjax", "adaptation", "chees_adaptation.py")
        if os.path.exists(_chees):
            with open(_chees, "r") as f:
                _csrc = f.read()
            _old_ci = "import blackjax.optimizers.dual_averaging as dual_averaging"
            _new_ci = "from blackjax.optimizers import dual_averaging"
            if _old_ci in _csrc:
                _csrc = _csrc.replace(_old_ci, _new_ci)
                with open(_chees, "w") as f:
                    f.write(_csrc)
                print("Patched chees_adaptation.py")
            break

    _gw_lik = os.path.join(sharpy_pkg, "GW_likelihood.py")
    _old_import = "from ripplegw import ms_to_Mc_eta"
    with open(_gw_lik, "r") as f:
        _src = f.read()
    if _old_import in _src and "try:" not in _src.split(_old_import)[0][-30:]:
        _new_import = (
            "try:\n"
            "    from ripplegw import ms_to_Mc_eta\n"
            "except ImportError:\n"
            "    def ms_to_Mc_eta(m):\n"
            "        m1, m2 = m\n"
            "        return (m1 * m2) ** (3 / 5) / (m1 + m2) ** (1 / 5), m1 * m2 / (m1 + m2) ** 2"
        )
        _src = _src.replace(_old_import, _new_import)
        with open(_gw_lik, "w") as f:
            f.write(_src)
        print("Patched GW_likelihood.py")

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")

elif IGWN:
    print("=== IGWN JupyterHub detected ===")
    print("Assuming mlgw-bns-jax conda environment is active.")
else:
    print("Local machine — skipping setup.")

In [ ]:
import os, sys, time
import numpy as np

_GPS_START = 1187008114
_DURATION  = 1024
_SRATE     = 4096
_DATA_DIR  = "gw170817_data"
os.makedirs(_DATA_DIR, exist_ok=True)

_DETECTORS = ["H1", "L1", "V1"]

_DCC_GWF_URL = (
    "https://dcc.ligo.org/public/0144/T1700406/003/"
    "L-L1_CLEANED_HOFT_C02_T1700406_v3-1187008667-4096.gwf"
)
_DCC_CHANNEL = "L1:DCH-CLEAN_STRAIN_C02_T1700406_v3"
_DCC_GPS0    = 1187008667
_DCC_SRATE   = 16384


def _ensure_gwf_backend():
    for mod in ("frameCPP", "lalframe", "framel"):
        try:
            __import__(mod)
            return
        except ImportError:
            pass
    import subprocess
    for pkg in ("framel",):
        ret = subprocess.call([sys.executable, "-m", "pip", "install", "-q", pkg])
        if ret == 0:
            return
    raise ImportError("Cannot read GWF files. Install framel.")


def _read_gwf_channel(path, channel, start, end):
    try:
        from gwpy.timeseries import TimeSeries
        ts = TimeSeries.read(path, channel, start=start, end=end)
        return np.asarray(ts.value, dtype=np.float64), float(ts.sample_rate.value)
    except Exception:
        pass
    import framel
    vec = framel.frgetvect1d(path, channel, start, end - start, 0)
    return np.asarray(vec[0], dtype=np.float64), 1.0 / vec[3]


_all_exist = all(
    os.path.isfile(os.path.join(_DATA_DIR,
        f"{d[0]}-{d}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt"))
    for d in _DETECTORS
)

if _all_exist:
    print("Cleaned data files already exist.")
else:
    from gwpy.timeseries import TimeSeries
    from scipy.signal import decimate as _decimate

    for det in _DETECTORS:
        out_file = os.path.join(_DATA_DIR,
            f"{det[0]}-{det}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt")

        if det == "L1":
            print("L1: building cleaned timeseries")
            ts_raw = TimeSeries.fetch_open_data("L1", _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)
            gwf_local = os.path.join(_DATA_DIR, "L1_cleaned_bw_T1700406.gwf")
            if not os.path.isfile(gwf_local):
                import requests
                print("  Downloading BayesWave GWF from DCC...", flush=True)
                resp = requests.get(_DCC_GWF_URL, stream=True)
                resp.raise_for_status()
                with open(gwf_local, "wb") as fout:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        fout.write(chunk)
            _ensure_gwf_backend()
            _need_end = _GPS_START + _DURATION
            bw_data, bw_sr = _read_gwf_channel(gwf_local, _DCC_CHANNEL, _DCC_GPS0, _need_end)
            if int(round(bw_sr)) != _SRATE:
                factor = int(round(bw_sr)) // _SRATE
                bw_data = _decimate(bw_data, factor, ftype="iir", zero_phase=True)
            n_raw = int((_DCC_GPS0 - _GPS_START) * _SRATE)
            strain = np.concatenate([ts_raw.value[:n_raw], bw_data])
            with open(out_file, "w") as fw:
                fw.write(f"# BayesWave-cleaned L1 strain GW170817\n# {_SRATE} Hz\n")
                for val in strain:
                    fw.write(f"{val:.16e}\n")
            print("  L1 done.")
        else:
            print(f"{det}: downloading from GWOSC...", flush=True)
            ts = TimeSeries.fetch_open_data(det, _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)
            with open(out_file, "w") as fw:
                fw.write(f"# {det} raw GWOSC strain GW170817\n# {_SRATE} Hz\n")
                for val in ts.value:
                    fw.write(f"{val:.16e}\n")
            print(f"  {det} done.")
    print("All detectors ready.")

In [ ]:
from __future__ import annotations
import os, sys, time
import numpy as np

if "google.colab" not in sys.modules:
    os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
print("JAX devices:", jax.devices())

In [ ]:
# ── Optimized: uses jax_predict_roq.py so PN is evaluated at O(N_ROQ)
# nodes instead of the model's internal O(6200)-pt grid. ──────────────
from roq_likelihood_mlgw_bns_jax import build_roq_template

MODEL_PATH = "mlgw_bns_jax_model.h5"
_template_nodes = build_roq_template(MODEL_PATH)

import sharpy.GW_likelihood as _gw_mod
from sharpy.GW_likelihood import GWNetwork
from sharpy.smc_functions import run_sharpy
import sharpy.PSDs

# Also patch _gw_mod.template so the full-grid sanity check uses the
# same model (just on the full grid instead of ROQ nodes).
from jax_predict_roq import load_predict_nodes as _lpn
_predict_nodes_full = _lpn(MODEL_PATH)

from sharpy.utils import McQ2Masses as _McQ2Masses

def _template_full_grid(params, frequency_array):
    """Same as _template_nodes but accepts an arbitrary frequency_array
    (used by the full-grid sanity-check likelihood)."""
    mc, q   = params[6], params[7]
    m1, m2  = _McQ2Masses(mc, q)
    mlgw_params = jnp.array([q, params[11], params[12], params[9], params[10]])
    hp, hc = _predict_nodes_full(
        mlgw_params, frequency_array,
        total_mass=m1 + m2,
        distance_mpc=jnp.exp(params[2]),
        inclination=params[3],
    )
    return hp * jnp.exp(-1j * params[4]), hc * jnp.exp(-1j * params[4])

_gw_mod.template = _template_full_grid

print("Model loaded. ROQ-optimised template (predict_nodes) ready.")
print("PN cost per call: O(N_query) — evaluated directly at query frequencies.")

In [ ]:
TRIGGER_TIME     = 1187008882.43
SEGMENT_DURATION = 128.0
SAMPLING_RATE    = 4096
F_LOWER          = 23.0
F_UPPER          = 2000.0
DATA_START_GPS   = 1187008114
DATA_DURATION    = 1024

FIXED_RA  = 3.44616     # rad (NGC 4993)
FIXED_DEC = -0.408084   # rad

DATA_DIR = "gw170817_data"
OUTDIR   = "outdir_GW170817_roq_optimized"
LABEL    = "GW170817_roq_pe_optimized"
os.makedirs(OUTDIR, exist_ok=True)

print(f"Segment: {SEGMENT_DURATION}s  ->  df = {1/SEGMENT_DURATION:.4f} Hz")
print(f"Fixed sky: RA={FIXED_RA:.5f}, Dec={FIXED_DEC:.6f} (NGC 4993)")

In [ ]:
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        f_lowr=F_LOWER, f_high=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

In [ ]:
from gwpy.timeseries import TimeSeries
import matplotlib.pyplot as plt

MERGER_GPS = TRIGGER_TIME
WINDOW = 6.0
T_START_PLOT = MERGER_GPS - WINDOW / 2
T_END_PLOT   = MERGER_GPS + WINDOW / 2
F_MIN_PLOT, F_MAX_PLOT = 20.0, 800.0
Q_RANGE = (4, 64)

DET_COLORS = {"H1": "Reds", "L1": "Blues", "V1": "Purples"}
DET_LABELS = {"H1": "LIGO Hanford (H1)", "L1": "LIGO Livingston (L1)", "V1": "Virgo (V1)"}

def _qtransform(filepath):
    strain = np.loadtxt(filepath, comments="#")
    ts = TimeSeries(strain, sample_rate=SAMPLING_RATE, t0=DATA_START_GPS)
    ts_w = ts.whiten(4, 2)
    ts_c = ts_w.crop(T_START_PLOT - 1, T_END_PLOT + 1)
    return ts_c.q_transform(frange=(F_MIN_PLOT, F_MAX_PLOT), qrange=Q_RANGE,
                            outseg=(T_START_PLOT, T_END_PLOT), logf=True)

raw_file = os.path.join(DATA_DIR,
    f"L-L1_GWOSC_4KHZ_R1-{DATA_START_GPS}-{DATA_DURATION}.txt")

if os.path.isfile(raw_file):
    qt_raw = _qtransform(raw_file)
    qt_cln = _qtransform(data_files["L1"])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
    for ax, qt, title in [(ax1, qt_raw, "L1 — Raw (with glitch)"),
                           (ax2, qt_cln, "L1 — BayesWave cleaned")]:
        pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                            qt.value.T, cmap="Blues", vmin=0, vmax=25)
        ax.set_yscale("log"); ax.set_ylim(F_MIN_PLOT, F_MAX_PLOT)
        ax.set_xlabel("Time relative to merger [s]", fontsize=13)
        ax.set_title(title, fontsize=14)
        ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7, label="Merger")
        ax.legend(loc="upper left"); ax.tick_params(labelsize=11)
        fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
    ax1.set_ylabel("Frequency [Hz]", fontsize=13)
    fig.suptitle("GW170817 — L1 glitch comparison", fontsize=15)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_L1_comparison.png"), dpi=150)
    plt.show()
else:
    print(f"Raw L1 file not found ({raw_file}) — skipping glitch comparison.")

fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
for ax, det in zip(axes, ["H1", "L1", "V1"]):
    qt = _qtransform(data_files[det])
    pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                        qt.value.T, cmap=DET_COLORS[det], vmin=0, vmax=25)
    ax.set_yscale("log"); ax.set_ylim(F_MIN_PLOT, F_MAX_PLOT)
    ax.set_ylabel("Frequency [Hz]", fontsize=13)
    ax.set_title(f"{DET_LABELS[det]} (BayesWave cleaned)", fontsize=13)
    ax.tick_params(labelsize=11)
    ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7)
    fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
axes[-1].set_xlabel("Time relative to merger [s]", fontsize=13)
fig.suptitle("GW170817 — Q-transform spectrograms (BayesWave-cleaned)", fontsize=15, y=0.995)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_qtransform_all.png"), dpi=150)
plt.show()
print("Spectrograms saved.")

## Load ROQ basis

Load the ROQ data products generated by `build_roq_basis_numpy.py` (or
`build_roq_basis_numpy.ipynb`).

At runtime we load **only `linear/`**. The quadratic basis is replaced
by a precomputed PSD-weighted **M-matrix** derived from the linear basis.

In [ ]:
ROQ_DIR = "./roq_basis_mlgw_bns_jax/ROQ_data"

B_lin_np  = np.load(os.path.join(ROQ_DIR, "linear", "basis_interpolant_linear.npy"))
nodes_lin = np.load(os.path.join(ROQ_DIR, "linear", "empirical_nodes_linear.npy"))
f_lin_np  = np.load(os.path.join(ROQ_DIR, "linear", "empirical_frequencies_linear.npy"))

f_lin_jax = jnp.array(f_lin_np, dtype=jnp.float64)
B_lin_jax = jnp.array(B_lin_np, dtype=jnp.complex128)

N_LIN = B_lin_np.shape[1]

DELTA_F = 1.0 / SEGMENT_DURATION
f_full = np.arange(F_LOWER, F_UPPER + DELTA_F, DELTA_F)

print(f"Full frequency grid : {len(f_full)} points")
print(f"Linear ROQ nodes    : {N_LIN} ({len(f_full)//N_LIN}x reduction)")
print(f"Waveform evaluations: {N_LIN} points (was {len(f_full)})")
print(f"B_lin shape: {B_lin_jax.shape}")

## ROQ Likelihood with tc-Grid

$$\ln \mathcal{L}_{\rm det} = \operatorname{Re}\!\left[\sum_j w_j(\tau)\,h_j\right]
- \frac{1}{2}\,\mathbf{h}^H M\mathbf{h} - \frac{1}{2}\langle d|d\rangle$$

The M-matrix for $\langle h|h\rangle$ is:

$$M_{jk} = \sum_f \overline{B}_{fj} \, \frac{4\Delta f}{S_n(f)} \, B_{fk}$$

In [ ]:
batched_det = gw_network.batched_detector
n_det = len(batched_det.Frequency)

N_TC_GRID = 3001
TC_MAX    = 0.15
tc_grid   = np.linspace(-TC_MAX, TC_MAX, N_TC_GRID)
tc_step   = tc_grid[1] - tc_grid[0]
tc_min    = tc_grid[0]

print(f"tc grid: {N_TC_GRID} points in [{-TC_MAX}, {TC_MAX}] s, step = {tc_step*1e3:.3f} ms")

data_lin_grid = np.zeros((n_det, N_TC_GRID, N_LIN), dtype=np.complex128)
M_hh_matrices = []
dd_terms = []

CHUNK = 100

t0_w = time.time()
for i in range(n_det):
    T_val   = float(batched_det.T[i])
    d_full  = np.array(batched_det.FrequencySeries[i])
    psd_val = np.array(batched_det.PowerSpectralDensity[i])
    inv_psd = 4.0 * DELTA_F / psd_val

    phase_T1  = np.exp(2j * np.pi * f_full * (T_val - 1.0))
    d_rotated = d_full * phase_T1
    d_rot_weighted = np.conj(d_rotated) * inv_psd

    for g0 in range(0, N_TC_GRID, CHUNK):
        g1 = min(g0 + CHUNK, N_TC_GRID)
        phase_chunk = np.exp(-2j * np.pi * np.outer(tc_grid[g0:g1], f_full))
        weighted_chunk = phase_chunk * d_rot_weighted[None, :]
        data_lin_grid[i, g0:g1] = weighted_chunk @ B_lin_np

    B_weighted = B_lin_np * np.sqrt(inv_psd)[:, None]
    M_i = B_weighted.conj().T @ B_weighted
    M_hh_matrices.append(M_i)

    dd_i = 4.0 * DELTA_F * np.sum(np.abs(d_full) ** 2 / psd_val)
    dd_terms.append(float(np.real(dd_i)))

    print(f"  Det {i}: (T-1)={T_val-1:.0f}s, <d|d>={dd_i:.2e}  ({time.time()-t0_w:.1f}s)")

data_lin_grid_jax = jnp.array(data_lin_grid)
M_hh_jax          = [jnp.array(M) for M in M_hh_matrices]
M_hh_stack        = jnp.stack(M_hh_jax, axis=0)
dd_jax            = jnp.array(dd_terms)
tc_grid_jax       = jnp.array(tc_grid)

print(f"\nROQ weights computed for {n_det} detectors in {time.time()-t0_w:.1f}s.")
print(f"  tc-grid data projection : {data_lin_grid_jax.shape}")
print(f"  M-matrix stack          : {M_hh_stack.shape}")

In [ ]:
# ── Build the optimized ROQ likelihood ────────────────────────────────
# Waveform PN is evaluated at O(N_ROQ) instead of O(N_model_grid=6200).
from roq_likelihood_mlgw_bns_jax import build_roq_likelihood

log_likelihood_roq_full, log_likelihood_roq_reduced = build_roq_likelihood(
    _template_nodes, f_lin_jax,
    n_det, batched_det,
    data_lin_grid_jax, M_hh_jax, dd_jax,
    tc_grid, N_TC_GRID,
    FIXED_RA=FIXED_RA, FIXED_DEC=FIXED_DEC,
)

print("ROQ likelihood functions defined (linear + M-matrix quadratic terms).")
print(f"  Waveform evaluated at {len(f_lin_jax)} ROQ nodes "
      f"(was {len(f_full)} — {len(f_full)//len(f_lin_jax)}x fewer).")
print(f"  PN cost per call: O({len(f_lin_jax)}) instead of O(6200) — true ROQ speedup.")
print(f"  M-matrix stack: {M_hh_stack.shape}")

In [ ]:
prior_bounds = jnp.array([
    [jnp.log(1.0),  jnp.log(75.0)],     # [0]  logdistance
    [0.0,           jnp.pi],             # [1]  inclination
    [0.0,           2 * jnp.pi],         # [2]  phic
    [0.0,           jnp.pi],             # [3]  pol
    [1.18,          1.21],               # [4]  mc
    [1.0,           2.0],                # [5]  q
    [-0.1,          0.1],                # [6]  tc
    [-0.5,          0.5],                # [7]  chi1
    [-0.5,          0.5],                # [8]  chi2
    [5.0,           5000.0],             # [9]  lambda_1
    [5.0,           5000.0],             # [10] lambda_2
])

boundary_conditions = jnp.array([0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0])

parameter_names = [
    "logdistance", "theta_jn", "phiref", "pol",
    "mc", "q", "tc", "chi1", "chi2", "lambda_1", "lambda_2",
]

def prior(params):
    return 0.0

print(f"Sampling {len(parameter_names)} parameters: {parameter_names}")

## Sanity check: ROQ vs full-grid likelihood

Both use `predict_nodes` (PN at O(N\_query) cost). The difference tests ROQ
accuracy, not a model difference.

In [ ]:
from sharpy.GW_likelihood import antenna_pattern_functions
from sharpy.utils import TimeDelayFromEarthCenter

f_full_jax = jnp.array(f_full, dtype=jnp.float64)
detector_data_jax = jnp.array(
    np.stack([np.array(batched_det.FrequencySeries[i]) for i in range(n_det)]),
    dtype=jnp.complex128,
)
psd_jax = jnp.array(
    np.stack([np.array(batched_det.PowerSpectralDensity[i]) for i in range(n_det)]),
    dtype=jnp.float64,
)

PSD_FLOOR = jnp.float64(1e-30)
DELTA_F_JAX = jnp.float64(DELTA_F)


def _full_logL_single_det(params_13, det_idx):
    """Full-grid log-likelihood using _template_full_grid (predict_nodes on full grid)."""
    lat  = batched_det.latitude[det_idx]
    lon  = batched_det.longitude[det_idx]
    gam  = batched_det.gamma[det_idx]
    zeta = batched_det.zeta[det_idx]
    elev = batched_det.elevation[det_idx]
    trig = batched_det.trigtime[det_idx]

    fplus, fcross = antenna_pattern_functions(params_13, lat, lon, gam, zeta, trig)

    ra, dec = params_13[0], params_13[1]
    tc = trig + params_13[8]
    timedelay = TimeDelayFromEarthCenter(lat, lon, elev, ra, dec, tc)
    timeshift = timedelay + params_13[8]

    hp_full, hc_full = _gw_mod.template(params_13, f_full_jax)
    time_phase = jnp.exp(-2j * jnp.pi * f_full_jax * (timeshift + (SEGMENT_DURATION - 1.0)))
    hp_full = hp_full * time_phase
    hc_full = hc_full * time_phase
    h_full = fplus * hp_full + fcross * hc_full

    d_full_det = detector_data_jax[det_idx]
    psd_det    = psd_jax[det_idx]
    inv_psd    = 4.0 * DELTA_F_JAX / jnp.maximum(psd_det, PSD_FLOOR)

    dh = jnp.real(jnp.sum(jnp.conj(d_full_det) * h_full * inv_psd))
    hh = jnp.real(jnp.sum(jnp.conj(h_full) * h_full * inv_psd))
    dd = jnp.real(jnp.sum(jnp.abs(d_full_det) ** 2 * inv_psd))

    return dh - 0.5 * hh - 0.5 * dd


def log_likelihood_full_grid_11(params_11):
    params_13 = jnp.array([
        FIXED_RA, FIXED_DEC,
        params_11[0],   # logdist
        params_11[1],   # incl
        params_11[2],   # phic
        params_11[3],   # pol
        params_11[4],   # mc
        params_11[5],   # q
        params_11[6],   # delta_tc
        params_11[7],   # chi1
        params_11[8],   # chi2
        params_11[9],   # lambda_1
        params_11[10],  # lambda_2
    ])
    logL = jnp.float64(0.0)
    for i in range(n_det):
        logL = logL + _full_logL_single_det(params_13, i)
    return logL


log_L_roq  = jax.jit(log_likelihood_roq_reduced)
log_L_full = jax.jit(log_likelihood_full_grid_11)

p_test = jnp.array([
    jnp.log(40.0), 2.5, 0.0, 0.0,
    1.186, 0.87, 0.0, 0.0, 0.0, 300.0, 300.0
])

_ = log_L_roq(p_test).block_until_ready()
_ = log_L_full(p_test).block_until_ready()

logL_roq  = float(log_L_roq(p_test))
logL_full = float(log_L_full(p_test))
dlogL     = logL_roq - logL_full

threshold = 1.0
mark    = "✓" if abs(dlogL) < threshold else "✗"
verdict = "good" if abs(dlogL) < threshold else "check"

print(f"ROQ  log-likelihood : {logL_roq:.2f}")
print(f"Full log-likelihood : {logL_full:.2f}")
print(f"Difference (ROQ − full) : {dlogL:.3f}   {mark} {verdict}  (|Δ| < {threshold:.1f})")

import timeit
n_eval = 50
t_roq  = timeit.timeit(lambda: log_L_roq(p_test).block_until_ready(),  number=n_eval)
t_full = timeit.timeit(lambda: log_L_full(p_test).block_until_ready(), number=n_eval)

print(f"\nROQ  timing ({n_eval} evals): {1000*t_roq/n_eval:.2f} ms/eval")
print(f"Full timing ({n_eval} evals): {1000*t_full/n_eval:.2f} ms/eval")
print(f"Speedup: ~{t_full/t_roq:.1f}x")

## JIT Warm-up

Pre-compile all SHARPy vmapped/NUTS kernels before the SMC loop.

In [ ]:
from sharpy.smc_functions import build_mass_matrix_fn, build_kernel_fn
from blackjax.mcmc import integrators
import blackjax

N_PARTICLES_WARMUP = 4

print("Warming up vmapped likelihood...", flush=True)
t0 = time.time()

prior_lo, prior_hi = prior_bounds[:, 0], prior_bounds[:, 1]

def prior_transform(u):
    return prior_lo + u * (prior_hi - prior_lo)

def log_likelihood_unit(u):
    return log_likelihood_roq_reduced(prior_transform(u))

def log_posterior_unit(u, beta=1.0):
    q = prior_transform(u)
    return beta * log_likelihood_roq_reduced(q) + prior(q)

vmapped_logL = jax.jit(jax.vmap(log_likelihood_unit))
dummy_u = jax.random.uniform(jax.random.PRNGKey(0),
                              shape=(N_PARTICLES_WARMUP, len(prior_bounds)))
_ = vmapped_logL(dummy_u).block_until_ready()
print(f"  vmapped likelihood compiled in {time.time()-t0:.1f}s")

print("Warming up mass matrix...", flush=True)
t0 = time.time()
prior_bounds_unit = jnp.array([[0.0, 1.0]] * prior_bounds.shape[0])
mass_matrix_fn = build_mass_matrix_fn(log_posterior_unit)
_ = mass_matrix_fn(dummy_u, 0.01)
print(f"  mass matrix compiled in {time.time()-t0:.1f}s")

print("Warming up NUTS kernel...", flush=True)
t0 = time.time()
kernel = blackjax.nuts.build_kernel(
    prior_bounds_unit, boundary_conditions,
    integrators.velocity_verlet, divergence_threshold=100,
)
kernel_fn = build_kernel_fn(kernel, log_posterior_unit, 0.3)
dummy_states = jax.vmap(blackjax.nuts.init, in_axes=(0, None))(
    dummy_u, lambda x: log_posterior_unit(x, 0.01),
)
dummy_keys   = jax.random.split(jax.random.PRNGKey(1), N_PARTICLES_WARMUP)
dummy_betas  = jnp.full(N_PARTICLES_WARMUP, 0.01)
dummy_metrics = jnp.tile(jnp.eye(len(prior_bounds)), (N_PARTICLES_WARMUP, 1, 1))
_ = kernel_fn(dummy_keys, dummy_states, dummy_betas, dummy_metrics)
print(f"  NUTS kernel compiled in {time.time()-t0:.1f}s")

print("\nAll JIT compilations done. SMC loop should run without stalls.")

In [ ]:
N_PARTICLES = 500
STEP_SIZE   = 0.3
ALPHA       = 0.95
SEED        = 42

print(f"Starting SHARPy SMC with optimized ROQ likelihood...")
print(f"  {N_PARTICLES} particles, {len(parameter_names)} parameters")
start = time.time()

result_dict = run_sharpy(
    log_likelihood_roq_reduced, prior,
    prior_bounds, boundary_conditions,
    ALPHA, N_PARTICLES, STEP_SIZE,
    jax.random.PRNGKey(SEED),
    folder=OUTDIR, label=LABEL,
)

dt = time.time() - start
samples = result_dict["posterior_samples"]
logZ, dlogZ = result_dict["logZ"], result_dict["dlogZ"]
print(f"\nDone in {dt:.1f}s ({dt/3600:.2f}h)")
print(f"log Z = {logZ:.2f} +/- {dlogZ:.2f}")
print(f"Posterior samples: {samples.shape}")

In [ ]:
from corner import corner

fig = corner(
    np.array(samples), show_titles=True,
    labels=parameter_names, title_kwargs={"fontsize": 10},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved: {plot_path}")
fig

In [ ]:
from sharpy.utils import McQ2Masses

mc_samples   = np.array(samples[:, 4])
q_samples    = np.array(samples[:, 5])
chi1_samples = np.array(samples[:, 7])
chi2_samples = np.array(samples[:, 8])
lam1_samples = np.array(samples[:, 9])
lam2_samples = np.array(samples[:, 10])
logd_samples = np.array(samples[:, 0])

m1_samples = np.zeros(len(mc_samples))
m2_samples = np.zeros(len(mc_samples))
for i in range(len(mc_samples)):
    m1_samples[i], m2_samples[i] = McQ2Masses(mc_samples[i], q_samples[i])

chi_eff_samples = (m1_samples * chi1_samples + m2_samples * chi2_samples) / (m1_samples + m2_samples)

M_samples = m1_samples + m2_samples
lambda_tilde_samples = (16.0 / 13.0) * (
    (m1_samples + 12.0 * m2_samples) * m1_samples**4 * lam1_samples
    + (m2_samples + 12.0 * m1_samples) * m2_samples**4 * lam2_samples
) / M_samples**5

dL_samples = np.exp(logd_samples)

paper_samples = np.column_stack([
    mc_samples, q_samples, chi_eff_samples,
    lambda_tilde_samples, dL_samples,
])

paper_labels = [
    r"$\mathcal{M}_c$ $[M_\odot]$",
    r"$q$",
    r"$\chi_{\rm eff}$",
    r"$\tilde{\Lambda}$",
    r"$D_L$ [Mpc]",
]

fig = corner(
    paper_samples, show_titles=True,
    labels=paper_labels,
    title_kwargs={"fontsize": 12},
    quantiles=[0.05, 0.5, 0.95],
    levels=(0.5, 0.9),
    fill_contours=True,
    color="tab:orange",
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner_paper.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved: {plot_path}")
fig